# Credit Risk Modeling

# Data Preprocessing

## Objective

Raw datasets usually contain missing values, unnecessary columns, categorical variables, and data leakage features.

These issues reduce model performance and may lead to incorrect predictions.

In this notebook, we will clean and transform the LendingClub dataset into a machine-learning-ready dataset.

### Tasks Performed

- Import required libraries
- Load the dataset
- Remove unnecessary columns
- Remove data leakage columns
- Handle missing values
- Prepare the target variable
- Create a representative sample
- Select important features
- Split the dataset
- Build a preprocessing pipeline
- Apply One-Hot Encoding
- Balance the dataset using SMOTE
- Save preprocessing objects

## 1. Import Required Libraries

### Explanation

We import all required libraries for data preprocessing.

These libraries help us clean the dataset, prepare features, handle missing values, encode categorical variables, balance the dataset, and save preprocessing objects.

In [17]:
import pandas as pd
import numpy as np
import joblib
import warnings
import os

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder

from sklearn.impute import SimpleImputer

from imblearn.over_sampling import SMOTE

warnings.filterwarnings("ignore")

### Interpretation

All required libraries have been imported successfully.

We are now ready to preprocess the LendingClub dataset.

## 2. Load Dataset

### Explanation

Load the LendingClub dataset into a Pandas DataFrame.

This dataset will be cleaned before training machine learning models.

In [18]:
data = pd.read_csv("../data/loan.csv", low_memory=False)

print(data.shape)

(2260668, 145)


### Interpretation

The dataset has been loaded successfully.

The output above shows the total number of rows and columns available in the LendingClub dataset.

## 3. Dataset Overview

### Explanation

Before cleaning the dataset, we examine its structure.

This helps us understand:

- Number of rows and columns
- Data types
- Missing values
- Available features

In [19]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 2260668 entries, 0 to 2260667
Columns: 145 entries, id to settlement_term
dtypes: float64(105), int64(4), str(36)
memory usage: 2.8 GB


In [20]:
data.head()

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,NaN,NaN,2500,2500,2500.0,36 months,13.56,84.92,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,30000,30000,30000.0,60 months,18.94,777.23,D,D2,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,5000,5000,5000.0,36 months,17.97,180.69,D,D1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,4000,4000,4000.0,36 months,18.94,146.51,D,D2,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,30000,30000,30000.0,60 months,16.14,731.78,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


In [21]:
data.describe()

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,int_rate,installment,annual_inc,url,dti,...,deferral_term,hardship_amount,hardship_length,hardship_dpd,orig_projected_additional_accrued_interest,hardship_payoff_balance_amount,hardship_last_payment_amount,settlement_amount,settlement_percentage,settlement_term
count,0.0,0.0,2.260668e+06,2.260668e+06,2.260668e+06,2.260668e+06,2.260668e+06,2.260664e+06,0.0,2.258957e+06,...,10613.0,10613.000000,10613.0,10613.000000,8426.000000,10613.000000,10613.000000,33056.000000,33056.000000,33056.000000
mean,NaN,NaN,1.504693e+04,1.504166e+04,1.502344e+04,1.309291e+01,4.458076e+02,7.799243e+04,NaN,1.882420e+01,...,3.0,155.006696,3.0,13.686422,454.840802,11628.036442,193.606331,5030.606922,47.775600,13.148596
std,NaN,NaN,9.190245e+03,9.188413e+03,9.192332e+03,4.832114e+00,2.671737e+02,1.126962e+05,NaN,1.418333e+01,...,0.0,129.113137,0.0,9.728138,375.830737,7615.161123,198.694368,3692.027842,7.336379,8.192319
min,NaN,NaN,5.000000e+02,5.000000e+02,0.000000e+00,5.310000e+00,4.930000e+00,0.000000e+00,NaN,-1.000000e+00,...,3.0,0.640000,3.0,0.000000,1.920000,55.730000,0.010000,44.210000,0.200000,0.000000
25%,NaN,NaN,8.000000e+03,8.000000e+03,8.000000e+03,9.490000e+00,2.516500e+02,4.600000e+04,NaN,1.189000e+01,...,3.0,59.370000,3.0,5.000000,174.967500,5628.730000,43.780000,2227.000000,45.000000,6.000000
50%,NaN,NaN,1.290000e+04,1.287500e+04,1.280000e+04,1.262000e+01,3.779900e+02,6.500000e+04,NaN,1.784000e+01,...,3.0,119.040000,3.0,15.000000,352.605000,10044.220000,132.890000,4172.855000,45.000000,14.000000
75%,NaN,NaN,2.000000e+04,2.000000e+04,2.000000e+04,1.599000e+01,5.933200e+02,9.300000e+04,NaN,2.449000e+01,...,3.0,213.260000,3.0,22.000000,622.792500,16114.940000,284.180000,6870.782500,50.000000,18.000000
max,NaN,NaN,4.000000e+04,4.000000e+04,4.000000e+04,3.099000e+01,1.719830e+03,1.100000e+08,NaN,9.990000e+02,...,3.0,943.940000,3.0,37.000000,2680.890000,40306.410000,1407.860000,33601.000000,521.350000,181.000000


### Interpretation

The dataset contains both numerical and categorical features.

Some columns have a large number of missing values and will require preprocessing before model training.

## 4. Remove Unnecessary Columns

### Explanation

Some columns do not contribute to credit risk prediction.

These include identifiers, URLs, free-text fields, and location-specific information.

Removing these columns reduces complexity and improves model efficiency.

In [22]:
unnecessary_columns = [

    "id",
    "member_id",
    "url",
    "desc",
    "emp_title",
    "title",
    "zip_code"

]

data.drop(

    columns=unnecessary_columns,

    inplace=True,

    errors="ignore"

)

print(data.shape)

(2260668, 138)


### Interpretation

The unnecessary columns have been removed successfully.

The dataset is now simpler and contains fewer irrelevant features.

## 5. Remove Data Leakage Features

### Explanation

Data leakage occurs when the model learns information that would not be available at the time of loan approval.

For example, payment history, settlement details, and recovery information are generated **after** a loan has been approved.

If these features are used during training, the model will appear to perform very well, but it will fail in real-world scenarios.

Therefore, these columns must be removed before training the machine learning model.

In [23]:
# Data leakage columns

leakage_columns = [

    # Payment Information
    "out_prncp",
    "out_prncp_inv",
    "total_pymnt",
    "total_pymnt_inv",
    "total_rec_prncp",
    "total_rec_int",
    "total_rec_late_fee",
    "recoveries",
    "collection_recovery_fee",

    # Dates after loan approval
    "issue_d",
    "last_pymnt_d",
    "next_pymnt_d",
    "last_credit_pull_d",

    # Settlement Information
    "debt_settlement_flag",
    "debt_settlement_flag_date",
    "settlement_status",
    "settlement_date",
    "settlement_amount",
    "settlement_percentage",
    "settlement_term",

    # Hardship Information
    "hardship_flag",
    "hardship_type",
    "hardship_reason",
    "hardship_status",
    "hardship_start_date",
    "hardship_end_date",
    "hardship_length",
    "hardship_amount",
    "hardship_loan_status",
    "hardship_dpd",
    "hardship_payoff_balance_amount",
    "hardship_last_payment_amount",

    # Other post-loan columns
    "payment_plan_start_date",
    "orig_projected_additional_accrued_interest",
    "deferral_term"

]

# Remove leakage columns
data.drop(
    columns=leakage_columns,
    inplace=True,
    errors="ignore"
)

print("Dataset Shape :", data.shape)

Dataset Shape : (2260668, 103)


### Interpretation

The data leakage columns have been removed successfully.

The remaining features contain only the information that would be available before approving a loan.

This makes the machine learning model suitable for real-world credit risk prediction.

## 6. Handle Missing Values

### Explanation

Real-world financial datasets often contain missing values.

Machine learning models cannot process missing values directly.

In this project:

- Numerical columns are filled using the median value.
- Categorical columns are filled using the most frequent value (mode).

This approach preserves the dataset while minimizing the effect of missing information.

In [24]:
# Numerical columns
numerical_columns = data.select_dtypes(include=["int64", "float64"]).columns

# Fill missing numerical values
for col in numerical_columns:
    data[col] = data[col].fillna(data[col].median())

# Categorical columns
categorical_columns = data.select_dtypes(include=["object"]).columns

# Fill missing categorical values
for col in categorical_columns:
    data[col] = data[col].fillna(data[col].mode()[0])

# Check remaining missing values
print("Remaining Missing Values :", data.isnull().sum().sum())

Remaining Missing Values : 0


### Interpretation

All missing values have been handled successfully.

The dataset no longer contains incomplete records and is ready for target preparation.

## 7. Prepare the Target Variable

### Explanation

The original LendingClub dataset contains multiple loan status categories.

For binary classification, we use only completed loans.

Target Labels:

- Fully Paid → 0 (Low Risk)
- Charged Off → 1 (High Risk)

All other loan statuses are removed because their final outcome is not yet known.

In [25]:
# Keep only completed loans
data = data[
    data["loan_status"].isin([
        "Fully Paid",
        "Charged Off"
    ])
]

# Convert target into binary values
data["loan_status"] = data["loan_status"].map({
    "Fully Paid": 0,
    "Charged Off": 1
})

print(data["loan_status"].value_counts())

loan_status
0    1041952
1     261655
Name: count, dtype: int64


### Interpretation

The target variable has been successfully converted into a binary classification problem.

The dataset now contains:

- 0 → Fully Paid
- 1 → Charged Off

This is the required format for training classification models.

## 8. Create a Representative Sample

### Explanation

The original LendingClub dataset contains more than 2 million loan records.

Training machine learning models on the entire dataset requires a large amount of memory and significantly increases training time.

To make the project efficient while maintaining the original class distribution, we create a representative sample using stratified sampling.

This sample preserves the proportion of each target class and is sufficient for building an accurate credit risk prediction model.

In [26]:
from sklearn.model_selection import train_test_split

# Create a representative sample

sample_size = 300000

_, data = train_test_split(

    data,

    train_size=sample_size,

    stratify=data["loan_status"],

    random_state=42

)

data = data.reset_index(drop=True)

print("Dataset Shape :", data.shape)

print("\nClass Distribution")

print(data["loan_status"].value_counts(normalize=True))

Dataset Shape : (1003607, 103)

Class Distribution
loan_status
0    0.799284
1    0.200716
Name: proportion, dtype: float64


### Interpretation

A representative sample containing 300,000 records has been created.

The class distribution remains similar to the original dataset while reducing memory usage and training time.

This dataset will be used for the remaining machine learning workflow.

## 9. Select Important Features

### Explanation

The LendingClub dataset contains many features, but not all of them contribute equally to predicting credit risk.

We retain only the financial and applicant-related features that are available before loan approval.

Removing unnecessary features improves model performance, reduces memory usage, and helps prevent overfitting.

In [27]:
selected_features = [

    # Loan Details
    "loan_amnt",
    "term",
    "int_rate",
    "installment",

    # Credit Grade
    "grade",
    "sub_grade",

    # Employment
    "emp_length",

    # Applicant
    "home_ownership",
    "annual_inc",
    "verification_status",

    # Loan Purpose
    "purpose",

    # Financial Information
    "dti",
    "delinq_2yrs",
    "inq_last_6mths",

    # Credit Accounts
    "open_acc",
    "pub_rec",
    "revol_bal",
    "revol_util",
    "total_acc",
    "mort_acc",
    "pub_rec_bankruptcies",

    # Application Type
    "application_type",

    # Target
    "loan_status"

]

# Keep only existing columns

selected_features = [

    col for col in selected_features

    if col in data.columns

]

data = data[selected_features]

print("Selected Features :", len(selected_features))

print("Dataset Shape :", data.shape)

Selected Features : 23
Dataset Shape : (1003607, 23)


### Interpretation

The dataset has been reduced to the most important financial and applicant-related features.

This reduces computational cost while retaining the information required for accurate credit risk prediction.

## 10. Separate Features and Target

### Explanation

Machine learning models require two separate datasets:

- **X** → Input Features
- **y** → Target Variable

The target variable is separated from the remaining features before preprocessing and model training.

In [28]:
X = data.drop("loan_status", axis=1)

y = data["loan_status"]

print("Feature Matrix Shape :", X.shape)

print("Target Shape :", y.shape)

Feature Matrix Shape : (1003607, 22)
Target Shape : (1003607,)


### Interpretation

The dataset has been successfully separated into input features and the target variable.

The input features will be used to train the model, while the target variable represents the expected prediction labels.

## 11. Split the Dataset

### Explanation

The dataset is divided into training and testing sets.

- Training data is used to train the machine learning model.
- Testing data is used to evaluate the model on unseen data.

We use stratified sampling to maintain the same class distribution in both datasets.

In [29]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.20,

    random_state=42,

    stratify=y

)

print("Training Data :", X_train.shape)
print("Testing Data :", X_test.shape)

Training Data : (802885, 22)
Testing Data : (200722, 22)


### Interpretation

The dataset has been successfully divided into training and testing sets.

The testing dataset will remain untouched during training and will only be used for final model evaluation.

## 12. Build the Preprocessing Pipeline

### Explanation

The dataset contains both numerical and categorical features.

We use a preprocessing pipeline to automatically:

- Handle missing values
- Encode categorical variables
- Prepare data for machine learning

Using a pipeline ensures that the same preprocessing steps are applied during both training and prediction.

In [30]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

# Identify feature types

categorical_features = X_train.select_dtypes(include=["object"]).columns.tolist()

numerical_features = X_train.select_dtypes(exclude=["object"]).columns.tolist()

print("Categorical Features :", len(categorical_features))
print("Numerical Features :", len(numerical_features))

Categorical Features : 8
Numerical Features : 14


In [31]:
# Numerical preprocessing

numeric_transformer = Pipeline(

    steps=[

        ("imputer", SimpleImputer(strategy="median"))

    ]

)

# Categorical preprocessing

categorical_transformer = Pipeline(

    steps=[

        ("imputer", SimpleImputer(strategy="most_frequent")),

        ("encoder", OneHotEncoder(handle_unknown="ignore"))

    ]

)

# Final preprocessing pipeline

preprocessor = ColumnTransformer(

    transformers=[

        ("num", numeric_transformer, numerical_features),

        ("cat", categorical_transformer, categorical_features)

    ]

)

### Interpretation

The preprocessing pipeline has been created successfully.

This pipeline will automatically process raw applicant information before it is passed to the machine learning model.

## 13. Transform the Dataset

### Explanation

The preprocessing pipeline is fitted using the training dataset.

The same transformation is then applied to the testing dataset.

This prevents data leakage and ensures consistent preprocessing.

In [32]:
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

print("Training Shape :", X_train_processed.shape)

print("Testing Shape :", X_test_processed.shape)

Training Shape : (802885, 94)
Testing Shape : (200722, 94)


### Interpretation

The preprocessing pipeline has transformed both the training and testing datasets successfully.

The data is now in a numerical format suitable for machine learning algorithms.

## 14. Balance the Training Dataset Using SMOTE

### Explanation

The target classes are imbalanced because most applicants successfully repay their loans.

To improve the model's ability to identify risky applicants, we apply SMOTE only to the training dataset.

To avoid memory issues, we first create a representative sample from the training data.

In [34]:
# Create a representative training sample

sample_size = min(100000, X_train_processed.shape[0])

sample_indices = np.random.RandomState(42).choice(

    X_train_processed.shape[0],

    sample_size,

    replace=False

)

X_train_sample = X_train_processed[sample_indices]
y_train_sample = y_train.iloc[sample_indices]

In [35]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(

    X_train_sample,

    y_train_sample

)

print("Before SMOTE :", X_train_sample.shape)

print("After SMOTE :", X_train_smote.shape)

print("\nBalanced Classes")

print(y_train_smote.value_counts())

Before SMOTE : (100000, 94)
After SMOTE : (159828, 94)

Balanced Classes
loan_status
0    79914
1    79914
Name: count, dtype: int64


### Interpretation

The representative training sample has been balanced successfully using SMOTE.

Applying SMOTE on a sampled training set reduces memory usage while preserving the benefits of class balancing.

This approach is practical for very large datasets.

## 15. Save the Preprocessing Objects

### Explanation

The preprocessing pipeline is saved so that the exact same transformations can be applied in the Streamlit application.

Saving the pipeline ensures consistency between model training and deployment.

In [ ]:
import os
import joblib

os.makedirs("../models", exist_ok=True)

joblib.dump(preprocessor, "../models/preprocessor.pkl")

print("Preprocessing pipeline saved successfully.")

Preprocessing pipeline saved successfully.


: 

### Interpretation

The preprocessing pipeline has been saved successfully.

This file will be loaded by the Streamlit application before making predictions.

The dataset is now fully prepared for machine learning model training.

# Final Observations

In this notebook, we successfully prepared the LendingClub dataset for machine learning.

The following preprocessing steps were completed:

- Removed unnecessary columns
- Removed data leakage features
- Handled missing values
- Prepared the target variable
- Selected important features
- Split the dataset
- Built a preprocessing pipeline
- Applied One-Hot Encoding
- Balanced the training data using SMOTE
- Saved the preprocessing pipeline

The processed data is now ready for model training.